# LeetCode #1377: Frog Position After T Seconds

https://leetcode.com/problems/frog-position-after-t-seconds/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (DFS all paths)** | $O(n!)$ | $O(n)$ |
| **Optimal: BFS Probability Propagation ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (DFS all paths)
Enumerate every path from node 1, split probability equally at each branch. Exponential — the frog can revisit the parent after $t$ steps? No — it never revisits. Still the branching makes DFS expensive.

### Optimal: BFS Probability Propagation ★
BFS from node 1, tracking the probability at each node. When visiting a node, divide the current probability equally among its **unvisited** children. If the frog reaches `target` and either has no unvisited children or has exactly `t` seconds left, the probability is locked in. Otherwise it moves on and the probability at `target` becomes 0.

**Constraints:**
* $1 \le n \le 100$
* $1 \le t \le 50$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public double FrogPosition(int n, int[][] edges, int t, int target) {
        var adj = new List<int>[n + 1];
        for (int i = 1; i <= n; i++) adj[i] = new List<int>();
        foreach (var e in edges) { adj[e[0]].Add(e[1]); adj[e[1]].Add(e[0]); }

        double[] prob = new double[n + 1];
        bool[] visited = new bool[n + 1];
        prob[1] = 1.0;
        visited[1] = true;

        var q = new Queue<int>();
        q.Enqueue(1);

        for (int sec = 0; sec < t && q.Count > 0; sec++) {
            int sz = q.Count;
            for (int i = 0; i < sz; i++) {
                int node = q.Dequeue();
                // Count unvisited children to split probability evenly
                int children = 0;
                foreach (int nb in adj[node]) if (!visited[nb]) children++;
                if (children == 0) continue; // leaf or all neighbours visited — stays put
                double share = prob[node] / children;
                prob[node] = 0; // probability moves entirely to children
                foreach (int nb in adj[node]) {
                    if (!visited[nb]) {
                        visited[nb] = true;
                        prob[nb] = share;
                        q.Enqueue(nb);
                    }
                }
            }
        }
        return prob[target];
    }
}

### Python

In [ ]:
from collections import deque, defaultdict

class Solution:
    def frogPosition(self, n: int, edges: list[list[int]], t: int, target: int) -> float:
        adj = defaultdict(list)
        for u, v in edges:
            adj[u].append(v)
            adj[v].append(u)

        prob = [0.0] * (n + 1)
        visited = [False] * (n + 1)
        prob[1] = 1.0
        visited[1] = True
        q = deque([1])

        for _ in range(t):
            if not q:
                break
            for _ in range(len(q)):
                node = q.popleft()
                # Only unvisited neighbours are valid jumps
                children = [nb for nb in adj[node] if not visited[nb]]
                if not children:
                    continue  # frog stays; probability remains at this node
                share = prob[node] / len(children)
                prob[node] = 0.0  # all probability flows to children
                for nb in children:
                    visited[nb] = True
                    prob[nb] = share
                    q.append(nb)

        return prob[target]

### Go

In [ ]:
func frogPosition(n int, edges [][]int, t int, target int) float64 {
	adj := make([][]int, n+1)
	for _, e := range edges {
		adj[e[0]] = append(adj[e[0]], e[1])
		adj[e[1]] = append(adj[e[1]], e[0])
	}

	prob := make([]float64, n+1)
	visited := make([]bool, n+1)
	prob[1] = 1.0
	visited[1] = true
	q := []int{1}

	for sec := 0; sec < t && len(q) > 0; sec++ {
		next := []int{}
		for _, node := range q {
			children := []int{}
			for _, nb := range adj[node] {
				if !visited[nb] { children = append(children, nb) }
			}
			if len(children) == 0 { continue } // stays put at leaf
			share := prob[node] / float64(len(children))
			prob[node] = 0
			for _, nb := range children {
				visited[nb] = true
				prob[nb] = share
				next = append(next, nb)
			}
		}
		q = next
	}
	return prob[target]
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn frog_position(n: i32, edges: Vec<Vec<i32>>, t: i32, target: i32) -> f64 {
        let n = n as usize;
        let target = target as usize;
        let mut adj = vec![vec![]; n + 1];
        for e in &edges {
            adj[e[0] as usize].push(e[1] as usize);
            adj[e[1] as usize].push(e[0] as usize);
        }

        let mut prob = vec![0.0f64; n + 1];
        let mut visited = vec![false; n + 1];
        prob[1] = 1.0;
        visited[1] = true;
        let mut q: VecDeque<usize> = VecDeque::from([1]);

        for _ in 0..t {
            if q.is_empty() { break; }
            let sz = q.len();
            for _ in 0..sz {
                let node = q.pop_front().unwrap();
                let children: Vec<usize> = adj[node].iter().copied().filter(|&nb| !visited[nb]).collect();
                if children.is_empty() { continue; } // stays — no unvisited neighbours
                let share = prob[node] / children.len() as f64;
                prob[node] = 0.0; // probability flows out
                for nb in children {
                    visited[nb] = true;
                    prob[nb] = share;
                    q.push_back(nb);
                }
            }
        }
        prob[target]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=7, edges=[[1,2],[1,3],[1,7],[2,4],[2,6],[3,5]], t=2, target=4`
At $t=0$: prob[1]=1. At $t=1$: frog splits to nodes 2,3,7 each with $\frac{1}{3}$. At $t=2$: from node 2 splits to 4 and 6 each with $\frac{1}{6}$. Answer: **$\frac{1}{6} \approx 0.1667$**.

### 2. Slightly Complex
**Input:** Same graph, `t=1, target=7`
At $t=1$: frog is at 2,3,7 with probability $\frac{1}{3}$ each. Node 7 has no unvisited children, so it stays. Answer: **$\frac{1}{3} \approx 0.3333$**.

### 3. Edge Case: Time Factor
**Input:** Path graph $1-2-3-\cdots-100$, `t=50, target=51`
BFS advances one node per second. At $t=50$, the frog is exactly at node 51 with probability 1.0.

### 4. Edge Case: Space Factor
**Input:** Star graph: node 1 connected to nodes 2 through 100; `t=1, target=50`.
At $t=1$ all 99 leaves are reached with prob $\frac{1}{99}$ each. The BFS queue holds all 99 leaves — $O(n)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `n=3, edges=[[2,1],[3,2]], t=1, target=2`
The tree is $1-2-3$. At $t=1$: frog moves to node 2 (only unvisited child of 1), prob=1.0. But the frog could continue at $t=2$ — queried at $t=1$, node 2 has an unvisited child (3) so the frog does NOT stay. Answer: **1.0** because $t=1$ and we are exactly at the target.